# Introduction
## Diffusion Equation
The 1D diffusion equation is given by
$$\frac{\partial u(x,t)}{\partial t} = \nu \frac{\partial^2 u(x,t)}{\partial x^2},$$
where
- $u(x,t)$ is the function we're tracking (velocity, temperature, etc)
- $x$ is position
- $t$ is time
- $\nu$ is the diffusion coefficient, ie how fast things spread

## Discretised Laplacian
The central second difference approximation
$$
\frac{d^2u(x_i)}{dx^2}\approx\frac{u_{i+1}-2u_i+u_{i-1}}{h^2}
$$
is used to form the Laplacian matrix
$$
L = \frac{1}{h^2}\begin{pmatrix}
-2 & 1 & 0 & 0 & 0 \\
1 & -2 & 1 & 0 & 0 \\
0 & 1 & -2 & 1 & 0 \\ 
0 & 0 & 1 & -2 & 1 \\
0 & 0 & 0 & 1 & -2
\end{pmatrix}
$$
where we have assumed dirichlet (non-periodic) boundary conditions.

## Time-Stepping Operator (CHANGE TO SECOND OR THIRD ORDER)
The time-stepping operator
$$
A = \mathbb I + \nu dt \  L
$$
is the operator that carries out the time evolution in steps using the Forward Euler scheme.

## Generating the Laplacian MPO


## Time Evolution using Tensor Networks
In the dense case, the state vector is updated by repeatedly applying the time-step operator
$$
\mathbf u(t_{i+1})=A  \mathbf u(t_i).
$$

In the tensor network formulation, $A$ and $\mathbf u(t_0)$ are converted into an MPO and an MPS respectively. The same update rule is performed in the tensor network language, which is to contract the MPO with the MPS to obtain the next update.

After each update, the resulting MPS is compressed via SVD truncation to limit bond dimension.

In [2]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import quimb.tensor as qtn
import time

# Helper Functions

In [ ]:
# ============
# DENSE SOLVER
# ============

def laplacian_1d_dense(N, dx):
    L = np.zeros((N, N))
    for i in range(N):
        L[i, i] = -2.0
        if i > 0:
            L[i, i-1] = 1.0
        if i < N-1:
            L[i, i+1] = 1.0
    return L / (dx * dx)

def laplacian_1d_sparse(N, dx):
    main = -2 * np.ones(N)
    off  = 1 * np.ones(N - 1)
    L = sp.diags(
        [off, main, off],
        offsets=[-1, 0, 1],
        format="csr"
    )
    return L / (dx * dx)


def evolve_dense(u0, steps, A, dt, save_every=50):
    u = u0.copy()

    saved = []
    times = []
    norms = []

    t = 0.0
    
    for i in range(steps):
        # save current state
        if i % save_every == 0:
            saved.append(u.copy())
            times.append(t)
            norms.append(np.linalg.norm(u))
    
        # advance one timestep
        u = A @ u
        t += dt
    
    # save final state
    saved.append(u.copy())
    times.append(t)
    norms.append(np.linalg.norm(u))
    
    return np.array(times), np.array(saved), np.array(norms)



# ==================
# MPS/MPO GENERATION
# ==================

# these helper functions convert from vectors to MPS and vice versa,
# as well as from matrices to MPOs

def vec_to_qtt_mps(u, n, cutoff=1e-10, max_bond=64):
    T = np.asarray(u).reshape((2,) * n)
    T = T.transpose(tuple(range(n - 1, -1, -1)))
    # reverses MPS direction such that least significant bit is first: easier for shift MPOs later
    return qtn.MatrixProductState.from_dense(T, cutoff=cutoff, max_bond=max_bond)


def qtt_mps_to_vec(mps):
    T = np.asarray(mps.to_dense())
    n = T.ndim
    T = T.transpose(tuple(range(n - 1, -1, -1)))
    # undo MPS direction reversal
    return T.reshape(-1)


def qtt_identity_mpo(n):

    W = np.zeros((1, 1, 2, 2))
    # creates a 4-dimensional tensor. \chi_left = \chi_right = 1, and input and output physical dims = 2
    # bond dims = 0 since each site acts independently

    W[0, 0] = np.eye(2)
    # fills the slice W[0,0,:,:] with an identity matrix, so now,
    # W[0,0,0,0] = 1
    # W[0,0,0,1] = 0
    # W[0,0,1,0] = 0
    # W[0,0,1,1] = 1
    # ie 0 goes to 0, 1 goes to 1

    arrays = [W.copy() for _ in range(n)] # duplicate n such tensors
    return qtn.MatrixProductOperator(arrays, shape='lrud') # l=left, r=right, u=upper/output, d=lower/input


def qtt_shift_plus_mpo(n):
    # we want an MPO that represents |i> -> |i+1>, except at the ends of course
    # this means the MPO looks like |x0 x1 x2 ... xn> -> |x0 x1 x2 ... xn> + 1, where each xk is 0 or 1, and the + is binary addition


    # so for example if we have 0101 + 1, we iterate through each bit starting from the least significant bit, ie rightmost
    # first bit: 1 + 1 = 0 with carry = 1

    # second bit: because carry = 1, we add 0 + 1 = 1
    # since there is no overflow, the carry resets to 0

    # third bit: carry = 0 so the bit remains unchanged
    # therefore all higher bits also remain unchanged


    # in MPS language, each site represents one bit, so 0101 corresponds to the state |0101>

    # the MPO implements this binary addition locally at each site
    # each tensor maps (input bit, carry_in) -> (output bit, carry_out)

    # the bond index stores the carry bit, so the bond dimension is 2
    # corresponding to carry = 0 or carry = 1

    # because vec_to_qtt_mps() reverses the ordering of bits,
    # the least significant bit is placed at the leftmost site

    # therefore the left boundary injects carry = 1 (adding one),
    # while the right boundary enforces carry = 0 (preventing overflow)


    arrays = []

    # first site: incoming carry fixed to 1
    W0 = np.zeros((1, 2, 2, 2))
    # [carry_in, carry_out, bit_out, bit_in]
    # leftmost site (corresponding to least significant bit) has left bond dim = 1 (carry_in must = 1) and right = 2 (one for carry=0, the other for carry=1)
    # input has two dimensions to receive the original bit and output has two dimensions to represent the result bit
    for bit_in in (0, 1):
        if bit_in == 0:
            # 0 + 1 = 1, carry = 0
            bit_out = 1
            carry = 0
        else:
            # 1 + 1 = 0 with carry
            bit_out = 0
            carry = 1
        W0[0, carry, bit_out, bit_in] = 1.0
    arrays.append(W0)

    # middle sites: propagate carry
    W = np.zeros((2, 2, 2, 2))  # [carry_in, carry_out, bit_out, bit_in]

    for carry_in in (0, 1):
        for bit_in in (0, 1):
            if carry_in == 0:
                # no carry coming in
                bit_out = bit_in
                carry_out = 0
            else:
                # carry_in = 1
                if bit_in == 0:
                    # 0 + 1 = 1
                    bit_out = 1
                    carry_out = 0
                else:
                    # 1 + 1 = 0 with carry
                    bit_out = 0
                    carry_out = 1
            W[carry_in, carry_out, bit_out, bit_in] = 1.0
    
    # actually this part can be expressed far more concisely as
    # bit_out   = bit_in ^ carry_in   (^ represents XOR in python)
    # carry_out = bit_in & carry_in   (& represents AND in python)

    for _ in range(n - 2):
        arrays.append(W.copy())

    # last site
    WN = np.zeros((2, 1, 2, 2))  # [carry_in, carry_out, bit_out, bit_in]

    for carry_in in (0, 1):
        for bit_in in (0, 1):
            if carry_in == 0:
                # no carry coming in
                bit_out = bit_in
                carry_out = 0
            else:
                # carry_in = 1
                if bit_in == 0:
                    # 0 + 1 = 1
                    bit_out = 1
                    carry_out = 0
                else:
                    # 1 + 1 = 0 with carry
                    bit_out = 0
                    carry_out = 1
            # only allow transitions where no overflow occurs
            if carry_out == 0:
                WN[carry_in, 0, bit_out, bit_in] = 1.0

    arrays.append(WN)

    return qtn.MatrixProductOperator(arrays, shape='lrud')


def qtt_shift_minus_mpo(n):
    arrays = []

    # first site
    W0 = np.zeros((1, 2, 2, 2)) # [borrow_in, borrow_out, bit_out, bit_in]
    # at the first site, borrow_in is fixed to 1 because we are subtracting 1
    for bit_in in (0, 1):
        if bit_in == 0:
            # 0 - 1 = 1 with borrow
            bit_out = 1
            borrow_out = 1
        else:
            # 1 - 1 = 0, no borrow
            bit_out = 0
            borrow_out = 0
        W0[0, borrow_out, bit_out, bit_in] = 1.0

    arrays.append(W0)

    # middle sites
    W = np.zeros((2, 2, 2, 2)) # [borrow_in, borrow_out, bit_out, bit_in]

    for borrow_in in (0, 1):
        for bit_in in (0, 1):

            if borrow_in == 0:
                # no borrow coming in, so the bit stays unchanged
                bit_out = bit_in
                borrow_out = 0

            else:
                # borrow_in = 1
                if bit_in == 0:
                    # 0 - 1 = 1 with borrow
                    bit_out = 1
                    borrow_out = 1
                else:
                    # 1 - 1 = 0, no further borrow
                    bit_out = 0
                    borrow_out = 0

            W[borrow_in, borrow_out, bit_out, bit_in] = 1.0

    for _ in range(n - 2):
        arrays.append(W.copy())

    # last site
    # at the final site we only allow borrow_out = 0
    WN = np.zeros((2, 1, 2, 2)) # [borrow_in, borrow_out, bit_out, bit_in]

    for borrow_in in (0, 1):
        for bit_in in (0, 1):

            if borrow_in == 0:
                # no borrow coming in, so the bit stays unchanged
                bit_out = bit_in
                borrow_out = 0

            else:
                # borrow_in = 1
                if bit_in == 0:
                    # 0 - 1 = 1 with borrow
                    bit_out = 1
                    borrow_out = 1
                else:
                    # 1 - 1 = 0, no further borrow
                    bit_out = 0
                    borrow_out = 0

            # only allow transitions with no underflow
            if borrow_out == 0:
                WN[borrow_in, 0, bit_out, bit_in] = 1.0
    
    arrays.append(WN)
    return qtn.MatrixProductOperator(arrays, shape='lrud')

def qtt_diffusion_mpo(n, alpha, cutoff=1e-12, max_bond=64):
    I  = qtt_identity_mpo(n)
    Sp = qtt_shift_plus_mpo(n)
    Sm = qtt_shift_minus_mpo(n)

    A = (1.0 - 2.0 * alpha) * I + alpha * Sp + alpha * Sm
    A.compress(cutoff=cutoff, max_bond=max_bond)
    return A


# ====================
# TIME EVOLUTION IN TN
# ====================

def step_mps(mps, mpo, cutoff=1e-10, max_bond=64):
    mps_new = mpo.apply(mps)
    mps_new.compress(cutoff=cutoff, max_bond=max_bond)
    return mps_new

def evolve_mps(mps0, mpoA, steps, save_every=50, cutoff=1e-10, max_bond=64):
    mps = mps0.copy()
    saved = []
    bonds = []
    
    for i in range(steps):
        t0 = time.perf_counter()
        if i % save_every == 0:
            saved.append(mps.copy())
            bonds.append(max(mps.bond_sizes()))
    
        mps = step_mps(mps, mpoA, cutoff, max_bond)
        print(i, max(mps.bond_sizes()), time.perf_counter() - t0)
    
    # save final state
    saved.append(mps.copy())
    bonds.append(max(mps.bond_sizes()))
    return saved, bonds



# =======================================
# WRAPPER FUNCTIONS TO MEASURE TIME TAKEN
# =======================================

def time_initialisation(N, dx, dt, nu, density="sparse"):
    t0 = time.perf_counter()

    if density == "sparse":
        L = laplacian_1d_sparse(N, dx)  
        A = sp.eye(N, format="csr") + dt * nu * L  
    elif density == "dense":
        L = laplacian_1d_dense(N, dx)
        A = np.eye(N) + dt * nu * L
    else:
        raise ValueError("density must be 'sparse' or 'dense'")
    
    t1 = time.perf_counter()
    return A, t1 - t0

def time_mps_construction(u0, n, cutoff, max_bond):
    t0 = time.perf_counter()
    mps0 = vec_to_qtt_mps(u0, n, cutoff, max_bond)
    t1 = time.perf_counter()
    return mps0, t1 - t0

def time_mpo_construction(n, alpha, mpo_cutoff, mpo_max_bond):
    t0 = time.perf_counter()
    mpoA = qtt_diffusion_mpo(n, alpha, cutoff=mpo_cutoff, max_bond=mpo_max_bond)
    t1 = time.perf_counter()
    return mpoA, t1 - t0

def time_evolve_dense(u0, steps, A, dt, save_every=50):
    t0 = time.perf_counter()
    times, us, norms = evolve_dense(u0, steps, A, dt, save_every)
    t1 = time.perf_counter()
    return times, us, norms, t1 - t0

def time_evolve_mps(mps0, mpoA, steps, save_every=50, cutoff=1e-10, max_bond=64):
    t0 = time.perf_counter()
    mps_saved, bond_track = evolve_mps(mps0, mpoA, steps, save_every, cutoff, max_bond)
    t1 = time.perf_counter()
    return mps_saved, bond_track, t1 - t0

# Parameters

In [ ]:
# ==================
# GENERAL PARAMETERS
# ==================

ns = [7]  
steps = 20               # number of steps required for time evolution
u0_type = "sines"      # "sines" | "quadratic" | "random"

nu = 1e-3                 # diffusion coefficient 
save_every = 200          # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1                 # controls time step relative to grid spacing. affects stability of time0-step scheme
density = "sparse"        # "sparse" | "dense", determines whether sparse or dense laplacian matrix is constructed. for sparse, max n is ~28. for dense, it is ~14



# =============
# TN PARAMETERS
# =============

# parameters for forming the initial MPS
init_cutoff = 1e-12
init_max_bond = 64

# parameters for forming the initial MPO
mpo_cutoff = 1e-12
mpo_max_bond = 256

# SVD truncation and bond dimension limits during contractions
tn_cutoff = 1e-10
tn_max_bond = 64

# Set Up

In [ ]:
n     = 15
N     = 2**n
x     = np.linspace(0, 1, N, endpoint=False)
dx    = x[1] - x[0] 
dt    = cfl * dx*dx / nu 
alpha = nu * dt / (dx * dx)

if u0_type == "sines":
    u0 = np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)
elif u0_type == "quadratic":
    u0 = x**2
elif u0_type == "random":
    u0 = np.random.randn(N)
else:
    u0 = np.asarray(u0_type)

In [ ]:
times_init  = []
times_mps   = []
times_mpo   = []
times_ev_d  = []
times_ev_tn = [] 

for n in ns:
    N     = 2**n
    x     = np.linspace(0, 1, N, endpoint=False)
    dx    = x[1] - x[0] 
    dt    = cfl * dx*dx / nu 
    alpha = nu * dt / (dx * dx)

    if u0_type == "sines":
        u0 = np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)
    elif u0_type == "quadratic":
        u0 = x**2
    elif u0_type == "random":
        u0 = np.random.randn(N)
    else:
        u0 = np.asarray(u0_type)

        

    A, t_init   = time_initialisation(N, dx, dt, nu, density) # initialise laplacian matrix L and time-stepping operator A

    mps0, t_mps = time_mps_construction(u0, n, init_cutoff, init_max_bond)
    mpoA, t_mpo = time_mpo_construction(n, alpha, mpo_cutoff, mpo_max_bond)

    times, us, norms, t_evolve_dense    = time_evolve_dense(u0, steps, A, dt, save_every)
    mps_saved, bond_track, t_evolve_mps = time_evolve_mps(mps0, mpoA, steps, save_every, tn_cutoff, tn_max_bond)

    times_init.append(t_init)
    times_mps.append(t_mps)
    times_mpo.append(t_mpo)
    times_ev_d.append(t_evolve_dense)
    times_ev_tn.append(t_evolve_mps)


0 13 0.0037355420063249767
1 41 0.10727783298352733
2 64 0.06923983403248712
3 64 0.22930483298841864
4 64 0.24000725004589185
5 64 0.4238403750350699
6 64 0.9104909999878146
7 64 1.9186615000362508
8 64 3.4383785420213826
9 64 10.191519415995572
10 64 29.136174999992363
11 64 88.11975545802852


: 

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(ns, times_init, marker='o', label='Dense setup')
plt.plot(ns, times_mps, marker='o', label='MPS construction')
plt.plot(ns, times_mpo, marker='o', label='MPO construction')
plt.yscale('log')
plt.xlabel('n  (N = 2^n)')
plt.ylabel('Time (s)')
plt.title('Setup costs')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(ns, times_ev_d, marker='o', label='Dense evolution')
plt.plot(ns, times_ev_tn, marker='o', label='TN evolution')
plt.yscale('log')
plt.xlabel('n  (N = 2^n)')
plt.ylabel('Time (s)')
plt.title('Evolution costs')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
Ns = [2**n for n in ns]

plt.figure(figsize=(8, 5))
plt.plot(Ns, times_ev_d, marker='o', label='Dense evolution')
plt.plot(Ns, times_ev_tn, marker='o', label='TN evolution')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('N')
plt.ylabel('Time (s)')
plt.title('Evolution time vs grid size')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()